## 13. 会话：continue / resume / fork 与持久化

> 来源：[Work with sessions](https://code.claude.com/docs/en/agent-sdk/sessions)、[Persist sessions to external storage](https://code.claude.com/docs/en/agent-sdk/session-storage)

Session = SDK 在 agent 工作期间累积的对话历史（prompt、每次 tool call、tool result、每个 response），自动写盘。它持久化的是**对话**，不是文件系统——要回滚 agent 的文件改动得用 file checkpointing（独立功能，§21「生产化」）。


### 13.1 按应用形态选方案

| 你在构建 | 用什么 |
|---|---|
| 一次性任务，无 follow-up | 单次 `query()`，什么都不用管 |
| 单进程内多轮对话 | `ClaudeSDKClient`（自动跟踪 session，无需管 ID） |
| 进程重启后接着上次 | `continue_conversation=True`（接当前目录最近一次） |
| 恢复特定历史 session | 捕获 session ID，传 `resume` |
| 尝试另一条路又不丢原线 | `resume` + `fork_session=True` |
| 多主机 / serverless 部署 | `session_store` 外部存储 adapter |

三种延续方式的语义：

- **continue**：接当前目录里最近的 session，不用跟踪任何 ID，适合单对话应用。
- **resume**：接指定 session ID。多用户/多 session 场景必须用它。session ID 从 `ResultMessage.session_id` 拿（每个 result 都有，成功失败都在，所以触限后也能 resume 放宽限制继续）。
- **fork**：`resume=<id>` + `fork_session=True` 新建一个 session，起点是原历史的副本；原 session 不动，得到两条可独立 resume 的线。**fork 分支的是对话，不是文件系统**——fork 出的 agent 改了文件，所有 session 都看得到。

完整走查见下方 cell：捕获 session_id → resume 追问（"it" 靠上一轮的上下文才解析得出来）→ fork 探索另一条路线。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


async def demo_sessions():
    session_id = None
    # 第一段：分析任务，捕获 session_id
    async for m in query(
        prompt="Read the authentication module",
        options=ClaudeAgentOptions(allowed_tools=["Read", "Glob"]),
    ):
        if isinstance(m, ResultMessage):
            session_id = m.session_id  # 成功失败都有，触限后也能 resume

    # resume：同一上下文继续，"it" 指代上一轮的 auth 模块
    async for m in query(
        prompt="Now find all places that call it",
        options=ClaudeAgentOptions(resume=session_id),
    ):
        if isinstance(m, ResultMessage) and m.subtype == "success":
            print(m.result)

    # fork：从同一起点分叉探索另一条路线，原 session 不受影响
    async for m in query(
        prompt="Instead, outline how OAuth2 would work for this module",
        options=ClaudeAgentOptions(resume=session_id, fork_session=True, max_turns=5),
    ):
        if isinstance(m, ResultMessage):
            print("forked session:", m.session_id)  # 与 session_id 不同


await demo_sessions()

### 13.2 存储位置与跨主机

Session 文件存于 `~/.claude/projects/<encoded-cwd>/*.jsonl`；`<encoded-cwd>` 是绝对工作目录路径把非字母数字字符全部替换成 `-`（`/Users/me/proj` → `-Users-me-proj`）。设了 `CLAUDE_CONFIG_DIR` 则挪到该目录下。

> [!warning] resume 出来是全新会话？
> 最常见原因是 `cwd` 不一致——session 文件按工作目录归档，从别的目录 resume 会找不到文件。跨主机（CI、容器、serverless）三条路：① 把 `.jsonl` 搬到新主机**相同路径**且 `cwd` 一致；② 不搬 transcript，把结论存成应用状态塞进新 session 的 prompt（官方认为通常更稳）；③ 用 `session_store`。


### 13.3 `session_store`：镜像到外部存储

`ClaudeAgentOptions(session_store=<adapter>)` 把 transcript **镜像**（不是替代——本地盘永远先写）到 S3/Redis/数据库，任何主机都能 resume。adapter 实现 `SessionStore` 协议——必选 `append` / `load`，可选 `list_sessions` / `delete` / `list_subkeys`。官方 Python 接口：

```python
class SessionKey(TypedDict):
    project_key: str            # 工作目录的文件系统安全编码（即 §13.2「存储位置」 的 <encoded-cwd>）
    session_id: str             # 会话 UUID
    subpath: NotRequired[str]   # 属于子 agent transcript 时才有，如 "subagents/agent-<id>"；
                                # 缺省 = 主对话。当作不透明的 key 后缀即可

class SessionStore(Protocol):
    # 必选
    async def append(self, key: SessionKey, entries: list[SessionStoreEntry]) -> None: ...
    async def load(self, key: SessionKey) -> list[SessionStoreEntry] | None: ...
    # load 的调用时机：设了 resume 时、CLI 子进程 spawn 前一次性读取（不是流式增量读）；
    # 未知会话返回 None

    # 可选方法，但"未实现"的表现各不相同：
    async def list_sessions(self, project_key: str) -> list[SessionStoreListEntry]: ...
    # ↑ 未实现时，带 store 的 list_sessions() 调用和 continue_conversation 会**直接抛错**，
    #   不是静默降级
    async def delete(self, key: SessionKey) -> None: ...
    # ↑ 语义约定：删除主 key（无 subpath）必须级联删掉该 session 的全部 subkey；
    #   未实现时 deleteSession 是 no-op（适合 append-only 后端）
    async def list_subkeys(self, key) -> list[str]: ...
    # ↑ 未实现时 resume 只恢复主对话——"枚举有哪些子 agent"硬依赖它；
    #   按已知 subpath 直接读某个子 agent 的 transcript 则不依赖（§14.2.2「存储隔离与跨重启 resume」）
```

开发测试不用真存储，SDK 内置 `InMemorySessionStore`。下方 cell 是官方 quick start：第一次 `query()` 挂上 store，第二次带 `resume` 从 store 加载——第二问能接住第一问的上下文即验证成功。

In [ ]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    InMemorySessionStore,
    ResultMessage,
)

store = InMemorySessionStore()


async def demo_session_store():
    session_id = None
    # 第一次 query：transcript 边跑边镜像进 store
    async for message in query(
        prompt="List the Python files under src/",
        options=ClaudeAgentOptions(session_store=store),
    ):
        if isinstance(message, ResultMessage):
            session_id = message.session_id

    # 第二次 query：resume 时从 store 加载，带着第一次的完整上下文——
    # "those files" 只有接住上文才解析得出来
    async for message in query(
        prompt="Summarize what those files do",
        options=ClaudeAgentOptions(session_store=store, resume=session_id),
    ):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_session_store()

自定义 adapter 就是把 `append` / `load` 接到真实后端。Redis 骨架（客户端与序列化从略；官方在 TypeScript SDK 仓库 `examples/session-stores/` 提供 S3 / Redis / Postgres 参考实现）：

```python
class RedisSessionStore:
    def __init__(self, redis):
        self.redis = redis

    @staticmethod
    def _k(key):  # SessionKey → 存储键；subpath 缺省即主对话
        return f"{key['project_key']}/{key['session_id']}/{key.get('subpath', 'main')}"

    async def append(self, key, entries):
        for entry in entries:
            # 镜像写失败 SDK 重试（共至多 3 次尝试），重试可能重复投递——
            # 按 entry 的 uuid 去重是 adapter 的责任
            if await self.redis.sismember(self._k(key) + ":seen", entry["uuid"]):
                continue
            await self.redis.rpush(self._k(key), serialize(entry))
            await self.redis.sadd(self._k(key) + ":seen", entry["uuid"])
        # SDK 从不清理 store 数据——TTL/lifecycle 由 adapter 自己设
        await self.redis.expire(self._k(key), 30 * 24 * 3600)

    async def load(self, key):
        raw = await self.redis.lrange(self._k(key), 0, -1)
        return [deserialize(r) for r in raw] or None


options = ClaudeAgentOptions(
    session_store=RedisSessionStore(redis),
    session_store_flush="batched",   # 每 turn 刷一次；"eager" 每帧都刷
)
```

规则收束：

- **双写、本地盘永远是第一份**——store 只是镜像；不想让本地副本留下来（如多租户容器），在 `options.env` 里把 `CLAUDE_CONFIG_DIR` 指向临时目录即可。另外 store 只镜像 **transcript**：CLAUDE.md 记忆文件、工作目录里的产物都不归它管，需要跨主机就另行挂共享卷或对象存储同步。
- **镜像是 best-effort**——重试仍失败时该批被丢弃、agent 继续跑（本地盘已有全量），流里发一条 `MirrorErrorMessage`（§7.2「SystemMessage 家族」），要监控 store 丢数据就盯它；超时的调用不重试，因为原调用可能仍会落地。
- **entries 当不透明的 JSON 值**——按序存、按序还，deep-equal 即可，不要求字节相同（Postgres `jsonb` 重排 key 也没关系）。
- **两个不兼容**——`session_store` 不能与 `enable_file_checkpointing` 同用（checkpoint 备份只写本地盘、不进镜像，SDK 直接抛错）；TS 的 `persistSession: false` 同理。
- **fork 不是字节拷贝**——`fork_session` 会读出源 entries、重写其中的 session_id 和消息 UUID 再写入新 key；adapter 层的 CopyObject 捷径做不到，别自作聪明。
- **一致性测试**——`claude_agent_sdk.testing.run_session_store_conformance(MyStore)` 验证协议契约，可选方法未实现的用例自动跳过。

磁盘 session 的管理函数（同步函数，可用来构建 session 选择器、transcript 查看器）：

- `list_sessions(directory=None, limit=None, include_worktrees=True)` → `list[SDKSessionInfo]`：省略 `directory` 时跨全部项目查；结果按 `last_modified` **降序**排（第一个就是最新）；目录在 git 仓库内时 `include_worktrees=True` 会带上各 worktree 路径的会话。`SDKSessionInfo` 字段：`session_id`、`summary`（显示标题，取自定义标题 > 自动摘要 > 首条 prompt）、`last_modified` / `created_at`（epoch 毫秒）、`file_size`（远程存储后端为 `None`）、`custom_title`、`first_prompt`、`git_branch`、`cwd`、`tag`。
- `get_session_messages(session_id, directory=None, limit=None, offset=0)`：返回 compaction 后 agent 实际可见的消息链，要原始全量直接 `store.load()`。
- `get_session_info(session_id, directory=None)`：单会话查询，找不到返回 `None`。
- `rename_session(session_id, title)` / `tag_session(session_id, tag)`：重复调用安全、最后一次生效，`tag=None` 清除标签；`session_id` 不是合法 UUID 或 title/tag 为空抛 `ValueError`，会话不存在抛 `FileNotFoundError`。

（TypeScript 有 `persistSession: false` 可不落盘，Python 始终写盘。）